# LLM-as-judge stability check

Stability across 20 reproductions for personalization/tool_high JSON outputs.


### Setup
- Edit `base_dir` if you move the folder.
- `alpha` controls the significance level for tests.


In [12]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
from scipy import stats

# The notebook is in LLM_as_judge_score_repro directory, so use current directory
# If that doesn't work, try the relative path from workspace root
base_dir = Path.cwd()
if not (base_dir / "repro_1").exists():
    # Try relative path from workspace root
    base_dir = Path("gorilla/berkeley-function-call-leaderboard/LLM_as_judge_score_repro").resolve()

alpha = 0.05

# ===== Config: select models and files to read =====
# model_name_filter: model folder name to read, e.g. "google_gemini-3-flash-preview"
#                   If None, read results for all models
model_name_filter = "anthropic_claude-sonnet-4.5"  # e.g.: "google_gemini-3-flash-preview" or "anthropic_claude-opus-4.5" or "anthropic_claude-sonnet-4.5" or "moonshotai_kimi-k2-0905"

# file_name_filter: filename to read, e.g. "multi_turn_long_context_101_judge.json"
#                  If None, read all *_judge.json files
file_name_filter = None  # e.g.: "multi_turn_long_context_101_judge.json"
# ===========================================

print("Base dir:", base_dir)
print("Base dir exists:", base_dir.exists())
repro_dirs = list(base_dir.glob("repro_*"))
print(f"Repro dirs found: {len(repro_dirs)} directories")
if repro_dirs:
    print(f"First few: {[d.name for d in sorted(repro_dirs, key=lambda p: int(p.name.split('_')[-1]))[:5]]}")

print(f"\nFilter settings:")
print(f"  Model filter: {model_name_filter if model_name_filter else 'All models'}")
print(f"  File filter: {file_name_filter if file_name_filter else 'All *_judge.json files'}")


Base dir: <PROJECT_ROOT>/Desktop/Desktop - ADUAED19365LPMX/Agent_IX_Personalization/gorilla/berkeley-function-call-leaderboard/LLM_as_judge_score_repro
Base dir exists: True
Repro dirs found: 20 directories
First few: ['repro_1', 'repro_2', 'repro_3', 'repro_4', 'repro_5']

Filter settings:
  Model filter: anthropic_claude-sonnet-4.5
  File filter: All *_judge.json files


In [13]:

rows = []
for repro_dir in sorted(base_dir.glob("repro_*"), key=lambda p: int(p.name.split("_")[-1])):
    repro_idx = int(repro_dir.name.split("_")[-1])
    
    # If model_name_filter is set, first check that the model folder exists
    if model_name_filter:
        model_dir = repro_dir / model_name_filter
        if not model_dir.exists():
            print(f"Warning: Model '{model_name_filter}' not found in {repro_dir.name}, skipping...")
            continue
        search_dir = model_dir
    else:
        search_dir = repro_dir
    
    # Build the file search pattern
    if file_name_filter:
        # If a filename is specified, search that file directly
        search_pattern = f"**/{file_name_filter}"
    else:
        # Otherwise, search all *_judge.json files
        search_pattern = "**/*_judge.json"
    
    for path in search_dir.glob(search_pattern):
        # If model_name_filter is set, ensure the path is within the correct model folder
        if model_name_filter and model_name_filter not in str(path):
            continue
        
        # If file_name_filter is set, ensure the filename matches
        if file_name_filter and path.name != file_name_filter:
            continue
            
        try:
            with path.open() as f:
                data = json.load(f)
            parsed = data.get("parsed", {})
            for dim_name, dim_data in parsed.get("dimensions", {}).items():
                rows.append({
                    "repro": repro_idx,
                    "test_id": data.get("test_id"),
                    "dimension": dim_name,
                    "score": dim_data.get("score"),
                    "justification": dim_data.get("justification"),
                    "model_name": parsed.get("model_name"),
                    "persona": data.get("persona"),
                    "personalization": data.get("personalization"),
                    "path": str(path.relative_to(base_dir)),
                })
        except Exception as e:
            print(f"Error reading {path}: {e}")
            continue

df = pd.DataFrame(rows)
print(f"Total rows collected: {len(rows)}")
print(f"DataFrame shape: {df.shape}")
print(f"DataFrame columns: {list(df.columns)}")

if len(df) > 0 and all(col in df.columns for col in ["dimension", "test_id", "repro"]):
    df = df.sort_values(["dimension", "test_id", "repro"]).reset_index(drop=True)
    print("Total rows:", len(df))
    df.head()
else:
    print("Warning: DataFrame is empty or missing required columns!")
    if len(df) > 0:
        print("Available columns:", list(df.columns))


Total rows collected: 800
DataFrame shape: (800, 9)
DataFrame columns: ['repro', 'test_id', 'dimension', 'score', 'justification', 'model_name', 'persona', 'personalization', 'path']
Total rows: 800


In [14]:

print("Reproduction indexes:", sorted(df["repro"].unique().tolist()))
print("Dimensions:", sorted(df["dimension"].unique().tolist()))
print("Tests per reproduction:")
display(df.groupby("repro")["test_id"].nunique())


Reproduction indexes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Dimensions: ['commitment_consistency', 'initiative_timing', 'intent_alignment_drift', 'interaction_coherence', 'interaction_efficiency', 'interaction_preference_alignment', 'overall_user_experience', 'user_cognitive_load_trajectory']
Tests per reproduction:


repro
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
11    5
12    5
13    5
14    5
15    5
16    5
17    5
18    5
19    5
20    5
Name: test_id, dtype: int64


### Basic statistics per test/dimension
- n, mean, std, min, max, median
- coefficient of variation (std/mean)
- 95% CI for the mean (t distribution)
- Saved to `stability_basic_stats.csv`


In [15]:

def mean_ci(series, alpha=0.05):
    series = series.dropna()
    if len(series) <= 1:
        return (np.nan, np.nan)
    ci_half = stats.sem(series) * stats.t.ppf(1 - alpha / 2, df=len(series) - 1)
    return (series.mean() - ci_half, series.mean() + ci_half)

stats_rows = []
for (test_id, dim), sub in df.groupby(["test_id", "dimension"]):
    scores = sub["score"].dropna()
    ci_low, ci_high = mean_ci(scores, alpha=0.05)
    stats_rows.append({
        "test_id": test_id,
        "dimension": dim,
        "n": len(scores),
        "mean": scores.mean(),
        "std": scores.std(ddof=1),
        "min": scores.min(),
        "max": scores.max(),
        "median": scores.median(),
        "cv": (scores.std(ddof=1) / scores.mean()) if scores.mean() else np.nan,
        "ci_lower": ci_low,
        "ci_upper": ci_high,
    })

stats_df = pd.DataFrame(stats_rows).sort_values(["test_id", "dimension"])
stats_path = base_dir / "stability_basic_stats.csv"
stats_df.to_csv(stats_path, index=False)
print("Saved basic stats to", stats_path)
stats_df.head()


Saved basic stats to <PROJECT_ROOT>/Desktop/Desktop - ADUAED19365LPMX/Agent_IX_Personalization/gorilla/berkeley-function-call-leaderboard/LLM_as_judge_score_repro/stability_basic_stats.csv


,test_id,dimension,n,mean,std,min,max,median,cv,ci_lower,ci_upper
0,multi_turn_long_context_101,commitment_consistency,20,5.0,0.000000,5,5,5.0,0.000000,5.000000,5.000000
1,multi_turn_long_context_101,initiative_timing,20,4.4,0.502625,4,5,4.0,0.114233,4.164764,4.635236
2,multi_turn_long_context_101,intent_alignment_drift,20,5.0,0.000000,5,5,5.0,0.000000,5.000000,5.000000
3,multi_turn_long_context_101,interaction_coherence,20,2.0,0.000000,2,2,2.0,0.000000,2.000000,2.000000
4,multi_turn_long_context_101,interaction_efficiency,20,2.0,0.000000,2,2,2.0,0.000000,2.000000,2.000000


In [ ]:
stats_df_list = [group for _, group in stats_df.groupby("test_id")]


In [23]:
mean_std_list = [df["std"].mean() for df in stats_df_list]
print("Mean std list for each stats_df:", mean_std_list)
print("Mean of std means across stats_df:", np.mean(mean_std_list))


每个 stats_df 的 std 平均值列表： [0.12565617248750865, 0.04579344356656541, 0.0, 0.0, 0.0]
所有 stats_df 的 std 平均值的平均值： 0.03428992321081481


In [17]:
stats_df_list[0]

,test_id,dimension,n,mean,std,min,max,median,cv,ci_lower,ci_upper
0,multi_turn_long_context_101,commitment_consistency,20,5.0,0.000000,5,5,5.0,0.000000,5.000000,5.000000
1,multi_turn_long_context_101,initiative_timing,20,4.4,0.502625,4,5,4.0,0.114233,4.164764,4.635236
2,multi_turn_long_context_101,intent_alignment_drift,20,5.0,0.000000,5,5,5.0,0.000000,5.000000,5.000000
3,multi_turn_long_context_101,interaction_coherence,20,2.0,0.000000,2,2,2.0,0.000000,2.000000,2.000000
4,multi_turn_long_context_101,interaction_efficiency,20,2.0,0.000000,2,2,2.0,0.000000,2.000000,2.000000
5,multi_turn_long_context_101,interaction_preference_alignment,20,5.0,0.000000,5,5,5.0,0.000000,5.000000,5.000000
6,multi_turn_long_context_101,overall_user_experience,20,4.6,0.502625,4,5,5.0,0.109266,4.364764,4.835236
7,multi_turn_long_context_101,user_cognitive_load_trajectory,20,3.0,0.000000,3,3,3.0,0.000000,3.000000,3.000000


In [18]:
stats_df_list[1]

,test_id,dimension,n,mean,std,min,max,median,cv,ci_lower,ci_upper
8,multi_turn_long_context_114,commitment_consistency,20,5.00,0.000000,5,5,5.0,0.000000,5.000000,5.000000
9,multi_turn_long_context_114,initiative_timing,20,5.00,0.000000,5,5,5.0,0.000000,5.000000,5.000000
10,multi_turn_long_context_114,intent_alignment_drift,20,5.00,0.000000,5,5,5.0,0.000000,5.000000,5.000000
11,multi_turn_long_context_114,interaction_coherence,20,2.85,0.366348,2,3,3.0,0.128543,2.678544,3.021456
12,multi_turn_long_context_114,interaction_efficiency,20,2.00,0.000000,2,2,2.0,0.000000,2.000000,2.000000
13,multi_turn_long_context_114,interaction_preference_alignment,20,5.00,0.000000,5,5,5.0,0.000000,5.000000,5.000000
14,multi_turn_long_context_114,overall_user_experience,20,5.00,0.000000,5,5,5.0,0.000000,5.000000,5.000000
15,multi_turn_long_context_114,user_cognitive_load_trajectory,20,5.00,0.000000,5,5,5.0,0.000000,5.000000,5.000000


In [19]:
stats_df_list[2]

,test_id,dimension,n,mean,std,min,max,median,cv,ci_lower,ci_upper
16,multi_turn_long_context_138,commitment_consistency,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
17,multi_turn_long_context_138,initiative_timing,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
18,multi_turn_long_context_138,intent_alignment_drift,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
19,multi_turn_long_context_138,interaction_coherence,20,4.0,0.0,4,4,4.0,0.0,4.0,4.0
20,multi_turn_long_context_138,interaction_efficiency,20,2.0,0.0,2,2,2.0,0.0,2.0,2.0
21,multi_turn_long_context_138,interaction_preference_alignment,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
22,multi_turn_long_context_138,overall_user_experience,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
23,multi_turn_long_context_138,user_cognitive_load_trajectory,20,4.0,0.0,4,4,4.0,0.0,4.0,4.0


In [20]:
stats_df_list[3]

,test_id,dimension,n,mean,std,min,max,median,cv,ci_lower,ci_upper
24,multi_turn_long_context_144,commitment_consistency,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
25,multi_turn_long_context_144,initiative_timing,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
26,multi_turn_long_context_144,intent_alignment_drift,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
27,multi_turn_long_context_144,interaction_coherence,20,2.0,0.0,2,2,2.0,0.0,2.0,2.0
28,multi_turn_long_context_144,interaction_efficiency,20,2.0,0.0,2,2,2.0,0.0,2.0,2.0
29,multi_turn_long_context_144,interaction_preference_alignment,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
30,multi_turn_long_context_144,overall_user_experience,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
31,multi_turn_long_context_144,user_cognitive_load_trajectory,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0


In [21]:
stats_df_list[4]

,test_id,dimension,n,mean,std,min,max,median,cv,ci_lower,ci_upper
32,multi_turn_long_context_157,commitment_consistency,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
33,multi_turn_long_context_157,initiative_timing,20,4.0,0.0,4,4,4.0,0.0,4.0,4.0
34,multi_turn_long_context_157,intent_alignment_drift,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
35,multi_turn_long_context_157,interaction_coherence,20,2.0,0.0,2,2,2.0,0.0,2.0,2.0
36,multi_turn_long_context_157,interaction_efficiency,20,2.0,0.0,2,2,2.0,0.0,2.0,2.0
37,multi_turn_long_context_157,interaction_preference_alignment,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
38,multi_turn_long_context_157,overall_user_experience,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0
39,multi_turn_long_context_157,user_cognitive_load_trajectory,20,5.0,0.0,5,5,5.0,0.0,5.0,5.0



### Significance by dimension across reproductions
- Non-parametric Kruskal–Wallis test using scores grouped by reproduction (across all test_ids for that dimension).
- `p_value < alpha` means scores differ by reproduction in a statistically significant way.
- Saved to `stability_kruskal_by_dimension.csv`.


In [22]:

sig_rows = []
for dim, sub in df.groupby("dimension"):
    groups = []
    repro_labels = []
    for repro, g in sub.groupby("repro"):
        scores = g["score"].dropna().values
        if len(scores):
            repro_labels.append(repro)
            groups.append(scores)
    if len(groups) >= 2:
        kw_stat, kw_p = stats.kruskal(*groups)
    else:
        kw_stat, kw_p = (np.nan, np.nan)
    sig_rows.append({
        "dimension": dim,
        "groups": len(groups),
        "kw_stat": kw_stat,
        "p_value": kw_p,
        "significant": kw_p < alpha if not np.isnan(kw_p) else np.nan,
    })

sig_df = pd.DataFrame(sig_rows).sort_values("p_value")
sig_path = base_dir / "stability_kruskal_by_dimension.csv"
sig_df.to_csv(sig_path, index=False)
print("Saved Kruskal results to", sig_path)
sig_df


ValueError: All numbers are identical in kruskal


### Optional: coefficient of variation by dimension
Lower CV implies more stability across all test_ids and reproductions.


In [ ]:

cv_df = df.groupby("dimension").agg(mean_score=("score", "mean"), std_score=("score", "std"))
cv_df["cv"] = cv_df["std_score"] / cv_df["mean_score"].replace({0: np.nan})
cv_df.sort_values("cv")
